# 🔥 Hybrid EPS Forecasting — LightGBM + DeepSeek Ensemble

**Strategy:**
```
LightGBM  →  Predicts EPS magnitude (strong on numbers)
    +
DeepSeek  →  Predicts direction + provides reasoning (strong on context)
    ↓
Ensemble  →  Weighted average of both predictions
```

---

## 🗂️ Pipeline Phases

| Phase | Description |
|---|---|
| **Phase 1** | Install dependencies |
| **Phase 2** | Imports & configuration |
| **Phase 3** | Feature extraction from JSONL |
| **Phase 4** | LightGBM training & prediction |
| **Phase 5** | Load DeepSeek predictions from JSON |
| **Phase 6** | Ensemble — combine both models |
| **Phase 7** | Evaluate all three (LightGBM, DeepSeek, Ensemble) |

---
## ⚡ Phase 1 — Install Dependencies

In [1]:
%%capture
!pip install lightgbm scikit-learn optuna shap
print('✅ Dependencies installed')

---
## 📦 Phase 2 — Imports & Configuration

In [9]:
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_PATH    = "/kaggle/input/datasets/mhariyan/eps-forecast-dataset/final_verified_dataset (1).jsonl"   # ← update if needed
DEEPSEEK_RESULTS= "/kaggle/working/inference_results.json"             # ← your DeepSeek output file
METRICS_OUT     = "/kaggle/working/hybrid_metrics.json"

# ── Ensemble weights (must sum to 1.0) ────────────────────────────────────────
LGBM_WEIGHT     = 0.5    # LightGBM contribution
DEEPSEEK_WEIGHT = 0.5    # DeepSeek contribution

# ── LightGBM cross-validation folds ──────────────────────────────────────────
N_FOLDS = 7

print('✅ Configuration loaded')
print(f'   LightGBM weight : {LGBM_WEIGHT}')
print(f'   DeepSeek weight : {DEEPSEEK_WEIGHT}')
print(f'   CV folds        : {N_FOLDS}')

✅ Configuration loaded
   LightGBM weight : 0.5
   DeepSeek weight : 0.5
   CV folds        : 7


---
## 🔧 Phase 3 — Feature Extraction from JSONL

Parses every sample in the JSONL file and extracts structured numerical features from the raw text.

### Features extracted per sample
- **Revenue features:** last value, YoY growth rates, trend direction
- **Margin features:** operating margin history, trend
- **EPS features:** last known EPS, EPS growth rate, EPS volatility
- **Balance sheet:** debt/equity ratio, cash ratio
- **Cash flow:** operating cash flow trend
- **Time features:** how many years of data available

In [10]:
# ── Parse a single value from text ────────────────────────────────────────────
def parse_value(text: str, key: str) -> float | None:
    """
    Extract numeric value for a given key from financial text.
    Handles both Billion and Million suffixes.
    Example: 'Revenue: 87.49 Billion' → 87.49
             'Revenue: 500.00 Million' → 0.5
    """
    pattern = rf'{key}:\s*(-?\d+(?:\.\d+)?)\s*(Billion|Million|B|M)?'
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        return None
    value  = float(match.group(1))
    suffix = (match.group(2) or '').lower()
    if suffix in ('million', 'm'):
        value /= 1000.0   # convert to Billion for consistency
    return value


def parse_actual_eps(output_text: str) -> float | None:
    """Extract ground truth EPS from the dataset output field."""
    match = re.search(
        r'\bforecast\b\s*[:\=]\s*(-?\d+(?:\.\d+)?)',
        output_text, re.IGNORECASE
    )
    return float(match.group(1)) if match else None


def extract_year_blocks(input_text: str) -> list:
    """
    Split input text into per-year blocks.
    Returns list of dicts, one per historical year.
    """
    # Split on year markers like '**Year 2022:**'
    blocks = re.split(r'\*\*Year\s*(\d{4}):\*\*', input_text)
    years  = []

    i = 1
    while i < len(blocks) - 1:
        year_num = int(blocks[i])
        year_txt = blocks[i + 1]

        # Skip forecast year (no data)
        if 'Forecast EPS' in year_txt:
            i += 2
            continue

        years.append({
            'year'         : year_num,
            'revenue'      : parse_value(year_txt, 'Revenue'),
            'op_income'    : parse_value(year_txt, 'Op Income'),
            'net_income'   : parse_value(year_txt, 'Net Income'),
            'eps'          : parse_value(year_txt, 'EPS'),
            'cash'         : parse_value(year_txt, 'Cash'),
            'debt'         : parse_value(year_txt, 'Debt'),
            'equity'       : parse_value(year_txt, 'Equity'),
            'op_cash_flow' : parse_value(year_txt, 'Op Cash Flow'),
        })
        i += 2

    return years


def safe_growth(curr, prev):
    """Calculate growth rate safely, returns None if division by zero."""
    if prev is None or curr is None or prev == 0:
        return None
    return (curr - prev) / abs(prev)


def extract_features(sample: dict) -> dict | None:
    """
    Full feature extraction pipeline for one sample.
    Returns a flat dict of features, or None if insufficient data.
    """
    years = extract_year_blocks(sample['input'])

    if len(years) < 2:
        return None   # need at least 2 years for growth rates

    last  = years[-1]   # most recent year
    prev  = years[-2]   # year before
    first = years[0]    # earliest year

    feats = {}

    # ── Revenue features ──────────────────────────────────────────────────────
    feats['revenue_last']        = last['revenue']
    feats['revenue_growth_1y']   = safe_growth(last['revenue'], prev['revenue'])
    feats['revenue_growth_all']  = safe_growth(last['revenue'], first['revenue'])

    if len(years) >= 3:
        prev2 = years[-3]
        feats['revenue_growth_2y'] = safe_growth(prev['revenue'], prev2['revenue'])
        # Is growth accelerating or decelerating?
        g1 = feats['revenue_growth_1y']
        g2 = feats['revenue_growth_2y']
        feats['revenue_accel'] = (g1 - g2) if (g1 and g2) else None
    else:
        feats['revenue_growth_2y'] = None
        feats['revenue_accel']     = None

    # ── Operating margin features ─────────────────────────────────────────────
    def op_margin(y):
        if y['revenue'] and y['revenue'] != 0 and y['op_income'] is not None:
            return y['op_income'] / y['revenue']
        return None

    feats['op_margin_last']   = op_margin(last)
    feats['op_margin_prev']   = op_margin(prev)
    feats['op_margin_change'] = (
        feats['op_margin_last'] - feats['op_margin_prev']
        if feats['op_margin_last'] is not None and feats['op_margin_prev'] is not None
        else None
    )
    # Average margin across all years
    margins = [op_margin(y) for y in years if op_margin(y) is not None]
    feats['op_margin_avg'] = np.mean(margins) if margins else None

    # ── EPS features ──────────────────────────────────────────────────────────
    feats['eps_last']       = last['eps']
    feats['eps_prev']       = prev['eps']
    feats['eps_growth_1y']  = safe_growth(last['eps'], prev['eps'])
    feats['eps_growth_all'] = safe_growth(last['eps'], first['eps'])

    all_eps = [y['eps'] for y in years if y['eps'] is not None]
    feats['eps_volatility'] = np.std(all_eps)  if len(all_eps) > 1 else None
    feats['eps_trend']      = np.polyfit(range(len(all_eps)), all_eps, 1)[0] if len(all_eps) > 1 else None

    # ── Net income features ───────────────────────────────────────────────────
    feats['net_income_last']   = last['net_income']
    feats['net_income_growth'] = safe_growth(last['net_income'], prev['net_income'])

    # Net margin
    if last['revenue'] and last['revenue'] != 0 and last['net_income'] is not None:
        feats['net_margin_last'] = last['net_income'] / last['revenue']
    else:
        feats['net_margin_last'] = None

    # ── Balance sheet features ────────────────────────────────────────────────
    feats['debt_last']   = last['debt']
    feats['equity_last'] = last['equity']
    feats['cash_last']   = last['cash']

    # Debt/equity ratio
    if last['equity'] and last['equity'] != 0 and last['debt'] is not None:
        feats['debt_equity_ratio'] = last['debt'] / last['equity']
    else:
        feats['debt_equity_ratio'] = None

    # Debt trend (is company paying down debt?)
    feats['debt_change'] = safe_growth(last['debt'], prev['debt'])

    # ── Cash flow features ────────────────────────────────────────────────────
    feats['op_cashflow_last']   = last['op_cash_flow']
    feats['op_cashflow_growth'] = safe_growth(last['op_cash_flow'], prev['op_cash_flow'])

    # Cash flow to net income ratio (earnings quality)
    if last['net_income'] and last['net_income'] != 0 and last['op_cash_flow'] is not None:
        feats['cashflow_quality'] = last['op_cash_flow'] / last['net_income']
    else:
        feats['cashflow_quality'] = None

    # ── Time features ─────────────────────────────────────────────────────────
    feats['n_years_data']  = len(years)
    feats['forecast_year'] = last['year'] + 1

    return feats


# ── Run extraction on full dataset ────────────────────────────────────────────
print('Extracting features...')

records = []
skipped = 0

with open(DATASET_PATH, 'r') as f:
    raw_samples = [json.loads(l) for l in f if l.strip()]

for i, sample in enumerate(raw_samples):
    actual = parse_actual_eps(sample['output'])
    if actual is None:
        skipped += 1
        continue

    feats = extract_features(sample)
    if feats is None:
        skipped += 1
        continue

    feats['actual_eps']  = actual
    feats['sample_index']= i
    feats['source_file'] = sample.get('source_file', '')
    records.append(feats)

df = pd.DataFrame(records)

print(f'✅ Feature extraction complete')
print(f'   Samples extracted : {len(df)}')
print(f'   Skipped           : {skipped}')
print(f'   Features          : {len(df.columns) - 3}')
print(f'\nFeature columns:')
feature_cols = [c for c in df.columns if c not in ['actual_eps', 'sample_index', 'source_file']]
for c in feature_cols:
    missing = df[c].isna().sum()
    print(f'   {c:<30} missing={missing}')

Extracting features...
✅ Feature extraction complete
   Samples extracted : 669
   Skipped           : 8
   Features          : 28

Feature columns:
   revenue_last                   missing=0
   revenue_growth_1y              missing=7
   revenue_growth_all             missing=6
   revenue_growth_2y              missing=344
   revenue_accel                  missing=347
   op_margin_last                 missing=10
   op_margin_prev                 missing=7
   op_margin_change               missing=11
   op_margin_avg                  missing=5
   eps_last                       missing=1
   eps_prev                       missing=1
   eps_growth_1y                  missing=7
   eps_growth_all                 missing=8
   eps_volatility                 missing=1
   eps_trend                      missing=1
   net_income_last                missing=0
   net_income_growth              missing=0
   net_margin_last                missing=10
   debt_last                      missing=0
   equit

---
## 🌲 Phase 4 — LightGBM Training & Prediction

Uses 5-fold cross-validation so every sample gets an out-of-fold prediction. This avoids data leakage — each prediction is made by a model that never saw that sample during training.

**Why LightGBM:**
- Handles missing values natively (no imputation needed)
- Robust to outliers with Huber loss
- Fast to train on tabular financial data
- Built-in feature importance via SHAP

In [11]:
# ── Prepare features and target ───────────────────────────────────────────────
feature_cols = [c for c in df.columns
                if c not in ['actual_eps', 'sample_index', 'source_file']]

X = df[feature_cols].values
y = df['actual_eps'].values

# ── LightGBM parameters ───────────────────────────────────────────────────────
lgbm_params = {
    'objective'        : 'huber',       # robust to EPS outliers
    'alpha'            : 0.9,           # Huber threshold
    'metric'           : 'mae',
    'n_estimators'     : 500,
    'learning_rate'    : 0.05,
    'num_leaves'       : 31,
    'max_depth'        : -1,
    'min_child_samples': 10,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

# ── 5-fold cross-validated out-of-fold predictions ───────────────────────────
print(f'Training LightGBM with {N_FOLDS}-fold CV...')

kf              = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(X))
fold_maes       = []
models          = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model_lgb = lgb.LGBMRegressor(**lgbm_params)
    model_lgb.fit(
        X_train, y_train,
        eval_set              = [(X_val, y_val)],
        callbacks             = [lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(period=-1)],
    )

    val_preds              = model_lgb.predict(X_val)
    oof_predictions[val_idx] = val_preds
    fold_mae               = mean_absolute_error(y_val, val_preds)
    fold_maes.append(fold_mae)
    models.append(model_lgb)

    print(f'   Fold {fold+1}/{N_FOLDS} — MAE: {fold_mae:.4f}')

df['lgbm_predicted'] = oof_predictions

print(f'\n✅ LightGBM training complete')
print(f'   CV MAE mean  : {np.mean(fold_maes):.4f}')
print(f'   CV MAE std   : {np.std(fold_maes):.4f}')

# ── Feature importance ────────────────────────────────────────────────────────
importances = np.mean(
    [m.feature_importances_ for m in models], axis=0
)
importance_df = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print(f'\nTop 10 most important features:')
print(importance_df.head(10).to_string(index=False))

Training LightGBM with 7-fold CV...
   Fold 1/7 — MAE: 21.3371
   Fold 2/7 — MAE: 19.7428
   Fold 3/7 — MAE: 15.8231
   Fold 4/7 — MAE: 20.1524
   Fold 5/7 — MAE: 17.7666
   Fold 6/7 — MAE: 14.9038
   Fold 7/7 — MAE: 24.4921

✅ LightGBM training complete
   CV MAE mean  : 19.1740
   CV MAE std   : 3.0633

Top 10 most important features:
           feature  importance
          eps_last 1743.571429
          eps_prev  952.428571
    eps_volatility  653.571429
       debt_change  544.571429
 debt_equity_ratio  540.857143
  cashflow_quality  540.285714
 revenue_growth_1y  538.142857
     eps_growth_1y  506.285714
revenue_growth_all  464.285714
    eps_growth_all  452.428571


---
## 🤖 Phase 5 — Load DeepSeek Predictions

Loads your existing `inference_results.json` from the DeepSeek inference run and aligns predictions with the LightGBM dataset by `sample_index`.

In [14]:
# ── Load DeepSeek results ─────────────────────────────────────────────────────
with open(DEEPSEEK_RESULTS, 'r') as f:
    ds_results = json.load(f)

# Build lookup: sample_index → predicted EPS
ds_lookup = {r['index']: r['predicted'] for r in ds_results}

print(f'✅ Loaded {len(ds_results)} DeepSeek predictions')

# ── Align with LightGBM dataset ───────────────────────────────────────────────
df['deepseek_predicted'] = df['sample_index'].map(ds_lookup)

n_matched   = df['deepseek_predicted'].notna().sum()
n_unmatched = df['deepseek_predicted'].isna().sum()

print(f'   Matched to LGBM dataset : {n_matched}')
print(f'   No DeepSeek prediction  : {n_unmatched} (rejected by DeepSeek)')
print(f'\nSample alignment check:')
print(df[['source_file', 'actual_eps', 'lgbm_predicted', 'deepseek_predicted']].head(10).to_string(index=False))

✅ Loaded 649 DeepSeek predictions
   Matched to LGBM dataset : 649
   No DeepSeek prediction  : 20 (rejected by DeepSeek)

Sample alignment check:
         source_file  actual_eps  lgbm_predicted  deepseek_predicted
  MANKIND.NS_RAW.csv       47.68       38.261766               32.00
  MANKIND.NS_RAW.csv       49.20       41.983909               52.00
   MANORG.NS_RAW.csv       14.60        6.651666                 NaN
MANUGRAPH.NS_RAW.csv       -6.53        2.502923               -5.00
MANUGRAPH.NS_RAW.csv       -8.78        1.224693               -7.00
 MANYAVAR.NS_RAW.csv      242.87       17.239150               18.00
MARALOVER.NS_RAW.csv       -2.35        2.212071               -4.00
MARALOVER.NS_RAW.csv       -5.83        1.517739               -2.35
 MARATHON.NS_RAW.csv       37.19       26.867660               19.00
   MARICO.NS_RAW.csv       11.43        9.772682               12.00


---
## 🔀 Phase 6 — Ensemble

Three ensemble strategies:

1. **Weighted Average** — `LGBM_WEIGHT × lgbm_pred + DEEPSEEK_WEIGHT × deepseek_pred`
2. **Direction-Corrected** — Use LightGBM magnitude but DeepSeek direction
3. **Adaptive** — Use DeepSeek when it's confident (close to LightGBM), else use LightGBM

For samples where DeepSeek has no prediction, falls back to LightGBM alone.

In [15]:
# ── Only evaluate on samples where both models have predictions ───────────────
both_mask = df['deepseek_predicted'].notna()
df_both   = df[both_mask].copy()
df_lgbm   = df.copy()  # all samples — LGBM always has predictions

print(f'Samples with BOTH predictions : {len(df_both)}')
print(f'Samples with LGBM only        : {len(df_lgbm)}')

# ── Strategy 1: Weighted Average ─────────────────────────────────────────────
df_both['ensemble_weighted'] = (
    LGBM_WEIGHT     * df_both['lgbm_predicted'] +
    DEEPSEEK_WEIGHT * df_both['deepseek_predicted']
)

# ── Strategy 2: Direction-Corrected ──────────────────────────────────────────
# Use LightGBM magnitude but override with DeepSeek direction
def direction_corrected(lgbm_pred, ds_pred, prev_eps):
    """
    If DeepSeek and LightGBM disagree on direction vs prev_eps,
    trust DeepSeek's direction but use LightGBM's magnitude.
    """
    if prev_eps is None or prev_eps == 0:
        return lgbm_pred   # fallback to LGBM if no prev EPS

    lgbm_dir = np.sign(lgbm_pred - prev_eps)
    ds_dir   = np.sign(ds_pred   - prev_eps)

    if lgbm_dir == ds_dir:
        # Both agree — use weighted average
        return LGBM_WEIGHT * lgbm_pred + DEEPSEEK_WEIGHT * ds_pred
    else:
        # Disagree — trust DeepSeek direction, use LightGBM magnitude
        magnitude = abs(lgbm_pred - prev_eps)
        return prev_eps + ds_dir * magnitude

df_both['ensemble_dir_corrected'] = df_both.apply(
    lambda r: direction_corrected(
        r['lgbm_predicted'],
        r['deepseek_predicted'],
        r.get('eps_prev')
    ), axis=1
)

# ── Strategy 3: Adaptive ──────────────────────────────────────────────────────
# When both models are close → average them
# When they disagree a lot  → trust LightGBM (more numerically grounded)
def adaptive_ensemble(lgbm_pred, ds_pred, threshold_pct=0.3):
    """
    If DeepSeek is within threshold_pct of LightGBM → average
    Otherwise → use LightGBM with small DeepSeek nudge
    """
    if lgbm_pred == 0:
        return (lgbm_pred + ds_pred) / 2
    relative_diff = abs(lgbm_pred - ds_pred) / abs(lgbm_pred)
    if relative_diff <= threshold_pct:
        return 0.5 * lgbm_pred + 0.5 * ds_pred   # close — trust both equally
    else:
        return 0.7 * lgbm_pred + 0.3 * ds_pred   # far apart — trust LGBM more

df_both['ensemble_adaptive'] = df_both.apply(
    lambda r: adaptive_ensemble(r['lgbm_predicted'], r['deepseek_predicted']),
    axis=1
)

print('\n✅ All ensemble strategies computed')
print(df_both[['source_file', 'actual_eps', 'lgbm_predicted',
               'deepseek_predicted', 'ensemble_weighted',
               'ensemble_adaptive']].head(8).to_string(index=False))

Samples with BOTH predictions : 649
Samples with LGBM only        : 669

✅ All ensemble strategies computed
         source_file  actual_eps  lgbm_predicted  deepseek_predicted  ensemble_weighted  ensemble_adaptive
  MANKIND.NS_RAW.csv       47.68       38.261766               32.00          35.130883          35.130883
  MANKIND.NS_RAW.csv       49.20       41.983909               52.00          46.991954          46.991954
MANUGRAPH.NS_RAW.csv       -6.53        2.502923               -5.00          -1.248539           0.252046
MANUGRAPH.NS_RAW.csv       -8.78        1.224693               -7.00          -2.887654          -1.242715
 MANYAVAR.NS_RAW.csv      242.87       17.239150               18.00          17.619575          17.619575
MARALOVER.NS_RAW.csv       -2.35        2.212071               -4.00          -0.893965           0.348449
MARALOVER.NS_RAW.csv       -5.83        1.517739               -2.35          -0.416131           0.357417
 MARATHON.NS_RAW.csv       37.19    

---
## 📊 Phase 7 — Full Evaluation

Evaluates all models side by side on the same sample set:
- LightGBM alone
- DeepSeek alone
- Ensemble (weighted average)
- Ensemble (direction-corrected)
- Ensemble (adaptive)

In [16]:
import numpy as np

def compute_metrics(actual, predicted, prev_eps=None):
    """Compute all 5 metrics for a set of predictions."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)

    mse  = np.mean((actual - predicted) ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(actual - predicted))

    nonzero = actual != 0
    mdape   = np.median(
        np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])
    ) * 100 if nonzero.any() else None

    maape = np.mean(
        np.arctan(np.abs((actual - predicted) / (np.abs(actual) + 1e-8)))
    ) * 100

    dir_acc = None
    if prev_eps is not None:
        prev = np.array(prev_eps, dtype=float)
        mask = (prev != 0) & ~np.isnan(prev)
        if mask.any():
            ad = np.sign(actual[mask]    - prev[mask])
            pd_ = np.sign(predicted[mask] - prev[mask])
            dir_acc = np.mean(ad == pd_) * 100

    return {
        'N'       : len(actual),
        'MSE'     : round(float(mse),   4),
        'RMSE'    : round(float(rmse),  4),
        'MAE'     : round(float(mae),   4),
        'MdAPE'   : round(float(mdape), 2) if mdape is not None else None,
        'MAAPE'   : round(float(maape), 2),
        'DA_pct'  : round(float(dir_acc), 2) if dir_acc is not None else None,
    }


# ── Evaluate all models on the SHARED set (both predictions exist) ────────────
y_true   = df_both['actual_eps'].values
prev_eps = df_both['eps_prev'].values

results = {
    'LightGBM'           : compute_metrics(y_true, df_both['lgbm_predicted'].values,          prev_eps),
    'DeepSeek'           : compute_metrics(y_true, df_both['deepseek_predicted'].values,       prev_eps),
    'Ensemble Weighted'  : compute_metrics(y_true, df_both['ensemble_weighted'].values,        prev_eps),
    'Ensemble Dir-Corr'  : compute_metrics(y_true, df_both['ensemble_dir_corrected'].values,   prev_eps),
    'Ensemble Adaptive'  : compute_metrics(y_true, df_both['ensemble_adaptive'].values,        prev_eps),
}

# Also evaluate LightGBM on its full set (all samples)
results['LightGBM (all)'] = compute_metrics(
    df_lgbm['actual_eps'].values,
    df_lgbm['lgbm_predicted'].values,
    df_lgbm['eps_prev'].values
)

# ── Print comparison table ────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  📊 FULL MODEL COMPARISON")
print(f"{'='*80}")
print(f"  {'Model':<25} {'N':>5} {'MAE':>8} {'RMSE':>8} {'MdAPE%':>8} {'MAAPE%':>8} {'DA%':>8}")
print(f"  {'─'*75}")

for model_name, m in results.items():
    da  = f"{m['DA_pct']:.2f}%"    if m['DA_pct']  is not None else 'N/A'
    mdp = f"{m['MdAPE']:.2f}%"     if m['MdAPE']   is not None else 'N/A'
    print(
        f"  {model_name:<25}"
        f"{m['N']:>5}"
        f"{m['MAE']:>8.2f}"
        f"{m['RMSE']:>8.2f}"
        f"{mdp:>9}"
        f"{m['MAAPE']:>8.2f}%"
        f"{da:>9}"
    )

print(f"{'='*80}")

# ── Find best model per metric ────────────────────────────────────────────────
# Exclude 'LightGBM (all)' from head-to-head since it uses different N
head_to_head = {k: v for k, v in results.items() if k != 'LightGBM (all)'}

best_mae  = min(head_to_head, key=lambda k: head_to_head[k]['MAE'])
best_da   = max(head_to_head, key=lambda k: head_to_head[k]['DA_pct'] or 0)
best_mdape= min(head_to_head, key=lambda k: head_to_head[k]['MdAPE']  or 999)

print(f"\n  🏆 Best MAE   : {best_mae}   ({head_to_head[best_mae]['MAE']:.2f})")
print(f"  🏆 Best MdAPE : {best_mdape}   ({head_to_head[best_mdape]['MdAPE']:.2f}%)")
print(f"  🏆 Best DA    : {best_da}   ({head_to_head[best_da]['DA_pct']:.2f}%)")

# ── Save full results ─────────────────────────────────────────────────────────
output = {
    'timestamp'      : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_both_samples' : len(df_both),
    'n_lgbm_total'   : len(df_lgbm),
    'ensemble_weights': {
        'lgbm'    : LGBM_WEIGHT,
        'deepseek': DEEPSEEK_WEIGHT,
    },
    'metrics'        : results,
    'best'           : {
        'mae'  : best_mae,
        'mdape': best_mdape,
        'da'   : best_da,
    }
}

with open(METRICS_OUT, 'w') as f:
    json.dump(output, f, indent=2)

# ── Save per-sample predictions CSV ──────────────────────────────────────────
df_both[[
    'source_file', 'actual_eps', 'eps_prev',
    'lgbm_predicted', 'deepseek_predicted',
    'ensemble_weighted', 'ensemble_dir_corrected', 'ensemble_adaptive'
]].to_csv('hybrid_predictions.csv', index=False)

print(f"\n  💾 Metrics saved to   : {METRICS_OUT}")
print(f"  💾 Predictions saved  : hybrid_predictions.csv")
print(f"  💾 Feature importance : use importance_df for analysis")


  📊 FULL MODEL COMPARISON
  Model                         N      MAE     RMSE   MdAPE%   MAAPE%      DA%
  ───────────────────────────────────────────────────────────────────────────
  LightGBM                   649   19.09   54.02   57.77%   61.85%   72.01%
  DeepSeek                   649   14.58   46.47   44.01%   49.19%   77.14%
  Ensemble Weighted          649   15.62   45.13   44.57%   53.82%   76.36%
  Ensemble Dir-Corr          649     nan     nan     nan%     nan%   77.14%
  Ensemble Adaptive          649   16.65   47.64   49.32%   56.74%   75.27%
  LightGBM (all)             669   19.17   53.63   59.15%   62.25%   71.60%

  🏆 Best MAE   : DeepSeek   (14.58)
  🏆 Best MdAPE : DeepSeek   (44.01%)
  🏆 Best DA    : DeepSeek   (77.14%)

  💾 Metrics saved to   : /kaggle/working/hybrid_metrics.json
  💾 Predictions saved  : hybrid_predictions.csv
  💾 Feature importance : use importance_df for analysis
